### Imports

In [79]:
import numpy as np

np.random.seed(42)

### Dataset and Normalization

In [80]:
X = np.array([[1,2,3],
              [2,3,4],
              [3,4,5],
              [4,5,6],
              [5,6,7]], dtype=float)

Y = np.array([[4],
              [5],
              [6],
              [7],
              [8]], dtype=float)

# Normalize the data
X = X / 10
Y = Y / 10

### Activation Functions

In [81]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def dsigmoid(y):
    return y * (1 - y)

def tanh(x):
    return np.tanh(x)

def dtanh(y):
    return 1 - y ** 2

### Loss Function

In [82]:
def loss_func(y_true, y_pred):
    return 0.5*np.mean((y_true - y_pred) ** 2)

### LSTM class

In [83]:
class LSTM:

    def __init__(self, input_size, hidden_size, output_size):

        self.input_size = input_size
        self.hidden_size = hidden_size

        concat_size = input_size + hidden_size

        # Forget gate weights
        self.Wf = np.random.randn(hidden_size, concat_size)*0.1
        self.bf = np.zeros((hidden_size, 1))

        # Input gate weights
        self.Wi = np.random.randn(hidden_size, concat_size)*0.1
        self.bi = np.zeros((hidden_size, 1))

        # Candidate memory cell weights
        self.Wc = np.random.randn(hidden_size, concat_size)*0.1
        self.bc = np.zeros((hidden_size, 1))

        # Output gate weights
        self.Wo = np.random.randn(hidden_size, concat_size)*0.1
        self.bo = np.zeros((hidden_size, 1))

        # output weights
        self.Wy = np.random.randn(output_size, hidden_size)*0.1
        self.by = np.zeros((output_size, 1))



    def forward(self, sequence):
        h = np.zeros((self.hidden_size, 1))
        c = np.zeros((self.hidden_size, 1))
        
        self.cache = []

        for x in sequence:
            x = x.reshape(-1, 1)
            z = np.vstack((h, x))
            f = sigmoid(self.Wf @ z + self.bf)
            i = sigmoid(self.Wi @ z + self.bi)
            g = tanh(self.Wc @ z + self.bc)
            o = sigmoid(self.Wo @ z + self.bo)
            
            c = f * c + i * g
            h = o * tanh(c)

            self.cache.append((z, f, i, g, o, c, h))
        
        y = self.Wy @ h + self.by
        return y
    

    def backward(self,target,lr):

        target = np.array([[target]])

        z,f,i,g,o,c,h = self.cache[-1]

        y = self.Wy@h + self.by
        dy = y-target

        dWy = dy@h.T
        dby = dy

        dh = self.Wy.T@dy

        dc = dh*o*dtanh(np.tanh(c))

        do = dh*np.tanh(c)*dsigmoid(o)

        di = dc*g*dsigmoid(i)

        dg = dc*i*dtanh(g)

        df = np.zeros_like(f)

        dWo = do@z.T
        dWi = di@z.T
        dWc = dg@z.T
        dWf = df@z.T

        self.Wy -= lr*dWy
        self.by -= lr*dby

        self.Wo -= lr*dWo
        self.bo -= lr*do

        self.Wi -= lr*dWi
        self.bi -= lr*di

        self.Wc -= lr*dWc
        self.bc -= lr*dg

        self.Wf -= lr*dWf
        self.bf -= lr*df
        

### Training

In [84]:
model = LSTM(input_size=1, hidden_size=8, output_size=1)

epochs = 1000
lr = 0.05
patience = 20
patience_count = 0
loss_history = []

for epoch in range(epochs):
    total_loss = 0
    
    for x,y in zip(X,Y):
        pred = model.forward(x)
        loss = loss_func(y, pred)
        total_loss += loss
        model.backward(y[0], lr)
    
    loss_history.append(total_loss)
    
    if len(loss_history) > 1 and total_loss > loss_history[-2]:
        patience_count += 1
    else:
        patience_count = 0

    if epoch%100 == 0:
        print(epoch, total_loss)

    if patience_count >= patience:
        print(f"Early stopping at epoch {epoch}")
        break


0 0.7972716276256764
100 0.05142732681684126
200 0.05008783137623842
300 0.04852706837639997
400 0.04660611819515444
500 0.044181894419971904
600 0.041114968900331714
700 0.03729127930558641
800 0.032661777972401565
900 0.027299703055011407


In [85]:
model.forward(np.array([10, 11, 12], dtype=float)/10)*10

array([[8.41259598]])